In [1]:
from pyspark.sql import SparkSession
import spark

In [2]:
spark = SparkSession.builder \
.appName ("Spark_Cache_Demo_RDD") \
.getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 14:21:53 INFO SparkEnv: Registering MapOutputTracker
26/04/14 14:21:53 INFO SparkEnv: Registering BlockManagerMaster
26/04/14 14:21:53 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/14 14:21:53 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
spark

In [4]:
!hadoop fs -ls /data/

Found 7 items
-rw-r--r--   2 root hadoop    1060750 2026-04-10 21:42 /data/customers.csv
-rw-r--r--   2 root hadoop       5488 2026-04-10 15:25 /data/customers_100.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-10 21:41 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-10 21:40 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop  236978176 2026-04-10 16:31 /data/customers_300mb.csv
-rw-r--r--   2 root hadoop        209 2026-04-11 17:27 /data/dates_data.csv
drwxr-xr-x   - root hadoop          0 2026-04-11 15:48 /data/write_output.csv


In [5]:
customers_rdd = spark.sparkContext.textFile('/data/customers_300mb.csv')

In [6]:
customers_filtered = customers_rdd.filter(lambda row:'Mumbai' in row)
customers_mapped = customers_filtered.map(lambda row: (row.split(',')[0],1))
customers_reduced = customers_mapped.reduceByKey(lambda x,y:x+y)

In [7]:
customers_reduced.count()

# 1st run --> 12 sec

457906

In [8]:
customers_reduced.getNumPartitions()

2

In [9]:
# 2nd Run 
 
customers_reduced.count()


# Visibly Faster # 1sec


457906

#### Caching

In [10]:
customers_reduced.cache() # Caching Is LAZY

PythonRDD[8] at RDD at PythonRDD.scala:53

In [11]:
# This time we are first caching it and then running our .Count()

customers_reduced.count()

# Took more time than last one ~ 1 sec

# In this first operation, it cached the data as well so some extra time.

457906

In [12]:

customers_reduced.count()


457906

#### To Uncache

In [13]:
customers_reduced.unpersist() # To Uncache

PythonRDD[8] at RDD at PythonRDD.scala:53

In [15]:
spark.stop()